In [1]:
!nvidia-smi

Fri Oct  9 05:10:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from numba import cuda
import os
device = cuda.get_current_device()
print(f"Device Name: {device.name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y}, {device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024, 64
Warp Size: 32


In [7]:
import numpy as np
import time
from numba import cuda

# ==========================================
# Task 2.1: Реализация ядер (Kernels)
# ==========================================

# 1. CPU Reference Implementation (Чистый цикл Python)
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

# 2. GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    # Расчет уникального одномерного ID потока
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

# ==========================================
# Task 2.2: Функция тестирования (Benchmark)
# ==========================================

def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    # 1. Измерение CPU Pure Loop (пропускаем для N >= 10M, чтобы не зависать)
    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan')

    # 2. Настройка GPU параметров
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Прогревочный запуск для компиляции (Warmup)
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    # 3. Полное время GPU раундтрипа (Включая пересылку по PCIe)
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    # 4. Время только вычисления на GPU (Данные уже в VRAM)
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

# ==========================================
# Запуск теста для всех порядков величин (N)
# ==========================================

# Массив тестовых размеров из вашей таблицы
sizes = [100, 1000, 10000, 100000, 1000000, 5000000, 10000000]

print(f"{'Array Size (N)':<15} | {'CPU Time (ms)':<15} | {'GPU Total (ms)':<15} | {'GPU Kernel Only (ms)':<20} | {'Winner':<10}")
print("-" * 85)

for N in sizes:
    cpu_t, gpu_tot, gpu_kern = benchmark_run(N, student_id_seed=1234) # Можете заменить сид на свой ID

    # Определяем победителя
    if np.isnan(cpu_t):
        winner = "GPU (CPU n/a)"
    elif cpu_t < gpu_tot:
        winner = "CPU"
    else:
        winner = "GPU"

    cpu_str = f"{cpu_t:.4f}" if not np.isnan(cpu_t) else "Skipped"
    print(f"{N:<15,} | {cpu_str:<15} | {gpu_tot:<15.4f} | {gpu_kern:<20.4f} | {winner:<10}")


Array Size (N)  | CPU Time (ms)   | GPU Total (ms)  | GPU Kernel Only (ms) | Winner    
-------------------------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


100             | 0.0756          | 1.2477          | 0.1151               | CPU       
1,000           | 8.4797          | 0.7903          | 0.0816               | GPU       
10,000          | 16.2307         | 4.9693          | 0.1051               | GPU       
100,000         | 97.0903         | 1.1645          | 0.1184               | GPU       
1,000,000       | 573.4863        | 4.7870          | 0.1225               | GPU       
5,000,000       | Skipped         | 13.1889         | 0.2814               | GPU (CPU n/a)
10,000,000      | Skipped         | 24.8859         | 0.4629               | GPU (CPU n/a)


In [10]:
# DEFECTIVE KERNEL: Notice the missing 'if idx < n' check!
@cuda.jit
def defective_kernel(d_arr):
 idx = cuda.grid(1)
 # FORBIDDEN: Writing without bounds check!
 d_arr[idx] = 999.0
# TEST HARNESS
N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)
# Launch 4 blocks of 256 = 1024 threads
threads_per_block = 256
blocks_per_grid = 4
print("Launching defective kernel with 1024 threads on 1000-element array...")
defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()
print("Execution finished (or did it crash?). Checking host copy...")
h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")


Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000. Last element: 999.0


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [12]:
import numpy as np
from numba import cuda

@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)

    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block # 5 блоков = 40 потоков

log_b = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_t = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_g = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_a = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()

print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<10}")
print("-" * 50)
for i in range(32, 40): # Печать критической границы перехода (Блок №4)
    status = "ACTIVE" if log_a.copy_to_host()[i] == 1 else "IDLE (GUARDED)"
    print(f"{log_g.copy_to_host()[i]:<10} | {log_b.copy_to_host()[i]:<10} | {log_t.copy_to_host()[i]:<10} | {status:<10}")


Global ID  | Block ID   | Thread ID  | Status    
--------------------------------------------------
32         | 4          | 0          | ACTIVE    
33         | 4          | 1          | ACTIVE    
34         | 4          | 2          | ACTIVE    
35         | 4          | 3          | ACTIVE    
36         | 4          | 4          | ACTIVE    
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [17]:
# BUGGY NAIVE SUMMATION
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
  idx = cuda.grid(1)
  if idx < d_arr.shape[0]:
    d_total[0] += d_arr[idx]
# CORRECT ATOMIC SUMMATION
@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
  idx = cuda.grid(1)
  if idx < d_arr.shape[0]:
 # HARDWARE ATOMIC LOCK: Hardware serializes updates to prevent collisions
    cuda.atomic.add(d_total, 0, d_arr[idx])


In [20]:
import numpy as np
from numba import cuda
import pandas as pd

# Naive summation kernel (contains a data race)
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)

    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]


# Correct summation kernel (uses atomic addition)
@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)

    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])


def main():
    # Experiment settings
    N = 50000
    TRIALS = 5
    THREADS_PER_BLOCK = 256
    TARGET_SUM = float(N)

    # Array of 50,000 ones
    h_arr = np.ones(N, dtype=np.float32)

    # Copy input data to GPU
    d_arr = cuda.to_device(h_arr)

    results = []

    for trial in range(1, TRIALS + 1):
        # Reset total to zero before every trial
        d_total = cuda.to_device(np.array([0.0], dtype=np.float32))

        # Launch naive kernel
        blocks = (N + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK
        naive_sum_kernel[blocks, THREADS_PER_BLOCK](d_arr, d_total)

        # Wait for GPU execution to finish
        cuda.synchronize()

        # Copy result back to CPU
        actual_sum = float(d_total.copy_to_host()[0])

        # Calculate lost updates
        deviation = TARGET_SUM - actual_sum

        results.append({
            "Trial": trial,
            "Target Sum": f"{TARGET_SUM:.1f}",
            "Naive Kernel Output": f"{actual_sum:.1f}",
            "Deviation / Lost Updates": f"{deviation:.1f}"
        })

    # Display results in an English table
    df = pd.DataFrame(results)

    print("\nNAIVE CUDA SUMMATION EXPERIMENT")
    print("=" * 90)
    print(df.to_string(index=False))
    print("=" * 90)

    # Optional: verify the correct atomic implementation
    d_total_atomic = cuda.to_device(
        np.array([0.0], dtype=np.float32)
    )

    atomic_sum_kernel[blocks, THREADS_PER_BLOCK](d_arr, d_total_atomic)
    cuda.synchronize()

    atomic_result = float(d_total_atomic.copy_to_host()[0])

    print(f"\nExpected Sum: {TARGET_SUM:.1f}")
    print(f"Atomic Kernel Output: {atomic_result:.1f}")
    print(f"Atomic Summation Correct: {np.isclose(atomic_result, TARGET_SUM)}")


if __name__ == "__main__":
    if not cuda.is_available():
        print("CUDA is not available. Please check your NVIDIA GPU, driver,")
        print("CUDA compatibility, and Numba installation.")
    else:
        main()


NAIVE CUDA SUMMATION EXPERIMENT
 Trial Target Sum Naive Kernel Output Deviation / Lost Updates
     1    50000.0                 2.0                  49998.0
     2    50000.0                 2.0                  49998.0
     3    50000.0                 2.0                  49998.0
     4    50000.0                 2.0                  49998.0
     5    50000.0                 2.0                  49998.0

Expected Sum: 50000.0
Atomic Kernel Output: 50000.0
Atomic Summation Correct: True


In [24]:

import numpy as np
import math
import hashlib
import pandas as pd
from numba import cuda


# ============================================================
# SECTION 5: 2D GRID IMAGE / MATRIX PROCESSING
# ============================================================

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    # Retrieve 2D coordinates
    x, y = cuda.grid(2)

    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)

        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(
            float(center_x**2 + center_y**2)
        )

        intensity = 1.0 - (dist / max_dist)

        if intensity < 0.0:
            intensity = 0.0

        d_canvas[y, x] = intensity


# Image dimensions
WIDTH, HEIGHT = 1024, 1024

# Initialize matrix
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

# Configure 2D blocks
threads_2d = (16, 16)

blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)

# Execute CUDA kernel
radial_vignette_2d[blocks_2d, threads_2d](
    d_canvas,
    WIDTH,
    HEIGHT,
    WIDTH // 2,
    HEIGHT // 2
)

cuda.synchronize()

# Copy GPU result to CPU
result_canvas = d_canvas.copy_to_host()

# Verify geometry
center_value = float(result_canvas[512, 512])
corner_value = float(result_canvas[0, 0])

geometry_results = pd.DataFrame([
    {
        "Verification": "Blocks in X",
        "Result": blocks_2d[0],
        "Expected": 64,
        "Status": "PASS" if blocks_2d[0] == 64 else "FAIL"
    },
    {
        "Verification": "Blocks in Y",
        "Result": blocks_2d[1],
        "Expected": 64,
        "Status": "PASS" if blocks_2d[1] == 64 else "FAIL"
    },
    {
        "Verification": "Total Blocks",
        "Result": blocks_2d[0] * blocks_2d[1],
        "Expected": 4096,
        "Status": (
            "PASS"
            if blocks_2d[0] * blocks_2d[1] == 4096
            else "FAIL"
        )
    },
    {
        "Verification": "Matrix Shape",
        "Result": str(result_canvas.shape),
        "Expected": "(1024, 1024)",
        "Status": (
            "PASS"
            if result_canvas.shape == (1024, 1024)
            else "FAIL"
        )
    },
    {
        "Verification": "Center Pixel (512, 512)",
        "Result": f"{center_value:.6f}",
        "Expected": "1.000000",
        "Status": (
            "PASS"
            if np.isclose(center_value, 1.0)
            else "FAIL"
        )
    },
    {
        "Verification": "Corner Pixel (0, 0)",
        "Result": f"{corner_value:.6f}",
        "Expected": "0.000000",
        "Status": (
            "PASS"
            if np.isclose(corner_value, 0.0, atol=1e-6)
            else "FAIL"
        )
    }
])

print("=" * 75)
print("SECTION 5: 2D GRID GEOMETRY VERIFICATION")
print("=" * 75)
print(geometry_results.to_string(index=False))
print("=" * 75)


# ============================================================
# SECTION 6: FINAL VERIFICATION CHECKSUM
# ============================================================

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()

    # Student ID
    h.update(student_id_str.strip().encode("utf-8"))

    # GPU name
    gpu_name = cuda.get_current_device().name
    if isinstance(gpu_name, str):
        gpu_name = gpu_name.encode("utf-8")
    h.update(gpu_name)

    # Sample of actual GPU computation output
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())

    # Crossover value from Section 2
    h.update(str(exp_table_crossover).encode("utf-8"))

    digest = h.hexdigest()[:16].upper()

    print("\n" + "=" * 60)
    print("SECTION 6: FINAL VERIFICATION CHECKSUM")
    print(f"GPU Name: {gpu_name.decode('utf-8', errors='replace')}")
    print(f"Student ID: {student_id_str}")
    print(f"Crossover N*: {exp_table_crossover}")
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)

    return digest


# Replace these with your actual details
STUDENT_ID = "230103108"
CROSSOVER_N = 100  # Replace with your measured Section 2 value

if not cuda.is_available():
    print("ERROR: CUDA is not available. Enable a GPU runtime in Colab.")
else:
    # Re-run kernel after verifying CUDA availability
    radial_vignette_2d[blocks_2d, threads_2d](
        d_canvas,
        WIDTH,
        HEIGHT,
        WIDTH // 2,
        HEIGHT // 2
    )
    cuda.synchronize()

    generate_lab_checksum(
        STUDENT_ID,
        d_canvas,
        CROSSOVER_N
    )


SECTION 5: 2D GRID GEOMETRY VERIFICATION
           Verification       Result     Expected Status
            Blocks in X           64           64   PASS
            Blocks in Y           64           64   PASS
           Total Blocks         4096         4096   PASS
           Matrix Shape (1024, 1024) (1024, 1024)   PASS
Center Pixel (512, 512)     1.000000     1.000000   PASS
    Corner Pixel (0, 0)     0.000000     0.000000   PASS

SECTION 6: FINAL VERIFICATION CHECKSUM
GPU Name: Tesla T4
Student ID: 230103108
Crossover N*: 100
OFFICIAL VERIFICATION CHECKSUM TOKEN: B0724D2790DE32C6
